# Exercise: Time Series Manipulations in Pandas

Like in prior exercises, you'll download a ticker's historical data from YahooFinance. But raw data isn't that useful by itself, so in this exercise you'll get practice with manipulating time series data in Pandas. 

In [ ]:
import pandas as pd
import yfinance as yf

**Pull data for your favorite ticker from YahooFinance**

Use the YF API to pull **daily** price data for at least 3 years for any stock ticker you'd like.

In [ ]:
stock_ticker = "000660.KS"
start_date = "2020-01-01"  # use format YYYY-MM-DD
end_date = "2026-05-11"

data = yf.download(
    tickers=stock_ticker, start=start_date, end=end_date
)  # replace ... inside this function with the correct parameters in order to get your data

In [ ]:
data.head(10)

**Resample from daily to weekly, taking the Friday price**

Using the above data, resample your `closing` price to weekly. In the demo, we used the last day of the period. In this exercise, can you figure out how to pull the weekly data that's for Friday specifically? 

In [ ]:
weekly_data_friday = data["Close"].resample("W-FRI").last()

In [ ]:
print(weekly_data_friday.tail(20))
print(data.tail(20))

**Taking time-based slice of your data**

Above you pulled the last 3 years of data for a given ticker. But in lots of trading applications, you don't need data that goes that far back in time as trends and underlying phenomenon have changed. 

Can you take just the last 3 months of data and put it into a new DataFrame? 

In [ ]:
three_months_ago = data.index.max() - pd.DateOffset(months=3)
three_months_ago

In [ ]:
last_3_months = data[data.index >= three_months_ago]
last_3_months.head()

In [ ]:
last_3_months.tail()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# --- Flatten to Series in case yfinance returns a single-column DataFrame ---
close = data["Close"].squeeze()
weekly = weekly_data_friday.squeeze().reindex(close.index)

# --- Moving averages ---
ma50 = close.rolling(window=50).mean()
ma200 = close.rolling(window=200).mean()

# --- RSI (14-period, Wilder smoothing via EWM) ---
delta = close.diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.ewm(com=13, adjust=False).mean()
avg_loss = loss.ewm(com=13, adjust=False).mean()
rsi = 100 - (100 / (1 + avg_gain / avg_loss))

# --- Plot layout: 2 rows, price on top, RSI on bottom ---
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)

ax1 = fig.add_subplot(gs[0])
ax1.plot(close.index, close, label="Daily Close", color="steelblue", linewidth=1)
ax1.plot(
    close.index,
    weekly,
    label="Weekly Close (Fri)",
    color="orange",
    linewidth=1.5,
    linestyle="--",
)
ax1.plot(close.index, ma50, label="50-day MA", color="green", linewidth=1.5)
ax1.plot(close.index, ma200, label="200-day MA", color="red", linewidth=1.5)
ax1.set_ylabel("Price")
ax1.set_title(f"{stock_ticker} — Daily/Weekly Close, 50/200-day MA, RSI(14)")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)
ax1.set_xticklabels([])

ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(rsi.index, rsi, label="RSI (14)", color="purple", linewidth=1)
ax2.axhline(70, color="red", linestyle="--", linewidth=0.8, label="Overbought (70)")
ax2.axhline(30, color="green", linestyle="--", linewidth=0.8, label="Oversold (30)")
ax2.fill_between(rsi.index, rsi, 70, where=(rsi >= 70), color="red", alpha=0.2)
ax2.fill_between(rsi.index, rsi, 30, where=(rsi <= 30), color="green", alpha=0.2)
ax2.set_ylim(0, 100)
ax2.set_ylabel("RSI")
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# --- Parameters ---
n_simulations = 1000
n_days = 252  # 1 trading year
np.random.seed(42)

# --- Calibrate from historical log returns ---
log_returns = np.log(close / close.shift(1)).dropna()
mu = log_returns.mean()  # daily drift
sigma = log_returns.std()  # daily volatility
last_price = close.iloc[-1]

# --- Monte Carlo via Geometric Brownian Motion ---
# S(t) = S0 * exp((mu - 0.5*sigma^2)*t + sigma*W(t))
dt = 1
drift = (mu - 0.5 * sigma**2) * dt
shocks = sigma * np.random.normal(0, 1, size=(n_days, n_simulations))
daily_returns = np.exp(drift + shocks)

price_paths = np.zeros((n_days + 1, n_simulations))
price_paths[0] = last_price
for t in range(1, n_days + 1):
    price_paths[t] = price_paths[t - 1] * daily_returns[t - 1]

final_prices = price_paths[-1]

# --- Scenario summary ---
scenarios = {
    "Extreme Bear (5th pct)": np.percentile(final_prices, 5),
    "Bear (25th pct)": np.percentile(final_prices, 25),
    "Base / Median (50th pct)": np.percentile(final_prices, 50),
    "Bull (75th pct)": np.percentile(final_prices, 75),
    "Extreme Bull (95th pct)": np.percentile(final_prices, 95),
}

print(f"Current price : {last_price:,.0f}")
print(f"Ann. drift    : {mu * 252:.1%}  |  Ann. volatility: {sigma * np.sqrt(252):.1%}")
print(f"\n{'Scenario':<30} {'Est. Price':>12} {'Change':>8}")
print("-" * 52)
for name, price in scenarios.items():
    chg = (price / last_price - 1) * 100
    print(f"{name:<30} {price:>12,.0f} {chg:>+7.1f}%")

# --- Plot ---
future_dates = pd.bdate_range(start=close.index[-1], periods=n_days + 1)

fig, ax = plt.subplots(figsize=(16, 7))

# Historical (last 2 years for readability)
two_years_ago = close.index.max() - pd.DateOffset(years=2)
hist = close[close.index >= two_years_ago]
ax.plot(hist.index, hist, color="steelblue", linewidth=1.2, label="Historical Close")

# Fan chart: percentile bands
pct_bands = [(5, 95, 0.10), (25, 75, 0.20), (40, 60, 0.30)]
for lo, hi, alpha in pct_bands:
    ax.fill_between(
        future_dates,
        np.percentile(price_paths, lo, axis=1),
        np.percentile(price_paths, hi, axis=1),
        color="orange",
        alpha=alpha,
        label=f"{lo}–{hi}th pct" if alpha == 0.10 else "_nolegend_",
    )

# Scenario lines
colors = {
    "Extreme Bear (5th pct)": "red",
    "Base / Median (50th pct)": "black",
    "Extreme Bull (95th pct)": "green",
}
for name, price in scenarios.items():
    if name in colors:
        ax.plot(future_dates[-1], price, "o", color=colors[name])
        ax.annotate(
            f"{name.split('(')[0].strip()}\n{price:,.0f}",
            xy=(future_dates[-1], price),
            xytext=(10, 0),
            textcoords="offset points",
            fontsize=8,
            color=colors[name],
            va="center",
        )
ax.plot(
    future_dates,
    np.percentile(price_paths, 50, axis=1),
    color="black",
    linewidth=1.5,
    linestyle="--",
    label="Median path",
)
ax.plot(
    future_dates,
    np.percentile(price_paths, 5, axis=1),
    color="red",
    linewidth=1,
    linestyle="--",
    label="5th pct (Extreme Bear)",
)
ax.plot(
    future_dates,
    np.percentile(price_paths, 95, axis=1),
    color="green",
    linewidth=1,
    linestyle="--",
    label="95th pct (Extreme Bull)",
)

ax.axvline(close.index[-1], color="grey", linestyle=":", linewidth=1)
ax.set_title(
    f"{stock_ticker} — 1-Year Monte Carlo Price Estimate ({n_simulations:,} simulations, GBM)"
)
ax.set_ylabel("Price")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Comparative Analysis — NVIDIA & ASML

Repeating the full analysis (price/MA/RSI chart + Monte Carlo 1-year forecast) for two major semiconductor stocks.

In [ ]:
def run_full_analysis(
    ticker,
    start="2020-01-01",
    end="2026-05-11",
    n_simulations=1000,
    n_days=252,
    seed=42,
):
    """Download data for `ticker` and produce:
    1. Price / MA50 / MA200 / RSI chart
    2. Monte Carlo 1-year forecast with scenario summary
    """
    np.random.seed(seed)

    # --- Data ---
    raw = yf.download(tickers=ticker, start=start, end=end, progress=False)
    close = raw["Close"].squeeze()
    weekly = close.resample("W-FRI").last().reindex(close.index)

    # --- Indicators ---
    ma50 = close.rolling(50).mean()
    ma200 = close.rolling(200).mean()
    delta = close.diff()
    avg_gain = delta.clip(lower=0).ewm(com=13, adjust=False).mean()
    avg_loss = (-delta.clip(upper=0)).ewm(com=13, adjust=False).mean()
    rsi = 100 - (100 / (1 + avg_gain / avg_loss))

    # ── Chart 1: Price + indicators ──────────────────────────────────────────
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.05)

    ax1 = fig.add_subplot(gs[0])
    ax1.plot(close.index, close, label="Daily Close", color="steelblue", lw=1)
    ax1.plot(
        close.index, weekly, label="Weekly Close (Fri)", color="orange", lw=1.5, ls="--"
    )
    ax1.plot(close.index, ma50, label="50-day MA", color="green", lw=1.5)
    ax1.plot(close.index, ma200, label="200-day MA", color="red", lw=1.5)
    ax1.set_ylabel("Price")
    ax1.set_title(f"{ticker} — Daily/Weekly Close, 50/200-day MA, RSI(14)")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)
    ax1.set_xticklabels([])

    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.plot(rsi.index, rsi, color="purple", lw=1, label="RSI (14)")
    ax2.axhline(70, color="red", ls="--", lw=0.8, label="Overbought (70)")
    ax2.axhline(30, color="green", ls="--", lw=0.8, label="Oversold (30)")
    ax2.fill_between(rsi.index, rsi, 70, where=(rsi >= 70), color="red", alpha=0.2)
    ax2.fill_between(rsi.index, rsi, 30, where=(rsi <= 30), color="green", alpha=0.2)
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("RSI")
    ax2.legend(loc="upper left")
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── Monte Carlo ──────────────────────────────────────────────────────────
    log_ret = np.log(close / close.shift(1)).dropna()
    mu = log_ret.mean()
    sigma = log_ret.std()
    last_px = close.iloc[-1]

    drift = mu - 0.5 * sigma**2
    shocks = sigma * np.random.normal(0, 1, size=(n_days, n_simulations))
    paths = np.zeros((n_days + 1, n_simulations))
    paths[0] = last_px
    daily_ret = np.exp(drift + shocks)
    for t in range(1, n_days + 1):
        paths[t] = paths[t - 1] * daily_ret[t - 1]

    final = paths[-1]
    scenarios = {
        "Extreme Bear (5th pct)": np.percentile(final, 5),
        "Bear (25th pct)": np.percentile(final, 25),
        "Base / Median (50th pct)": np.percentile(final, 50),
        "Bull (75th pct)": np.percentile(final, 75),
        "Extreme Bull (95th pct)": np.percentile(final, 95),
    }

    print(f"\n{'='*52}")
    print(f"  {ticker}  —  Monte Carlo 1-Year Forecast")
    print(f"{'='*52}")
    print(f"  Current price  : {last_px:,.2f}")
    print(
        f"  Ann. drift     : {mu * 252:.1%}   |   Ann. volatility: {sigma * np.sqrt(252):.1%}"
    )
    print(f"\n  {'Scenario':<28} {'Est. Price':>12} {'Change':>8}")
    print(f"  {'-'*50}")
    for name, price in scenarios.items():
        chg = (price / last_px - 1) * 100
        print(f"  {name:<28} {price:>12,.2f} {chg:>+7.1f}%")

    # Fan chart
    future_dates = pd.bdate_range(start=close.index[-1], periods=n_days + 1)
    two_yr_ago = close.index.max() - pd.DateOffset(years=2)
    hist = close[close.index >= two_yr_ago]

    fig2, ax = plt.subplots(figsize=(16, 7))
    ax.plot(hist.index, hist, color="steelblue", lw=1.2, label="Historical Close")
    for lo, hi, alpha in [(5, 95, 0.10), (25, 75, 0.20), (40, 60, 0.30)]:
        ax.fill_between(
            future_dates,
            np.percentile(paths, lo, axis=1),
            np.percentile(paths, hi, axis=1),
            color="orange",
            alpha=alpha,
            label=f"{lo}–{hi}th pct" if alpha == 0.10 else "_nolegend_",
        )
    ax.plot(
        future_dates,
        np.percentile(paths, 50, axis=1),
        color="black",
        lw=1.5,
        ls="--",
        label="Median path",
    )
    ax.plot(
        future_dates,
        np.percentile(paths, 5, axis=1),
        color="red",
        lw=1,
        ls="--",
        label="5th pct (Extreme Bear)",
    )
    ax.plot(
        future_dates,
        np.percentile(paths, 95, axis=1),
        color="green",
        lw=1,
        ls="--",
        label="95th pct (Extreme Bull)",
    )
    scenario_colors = {
        "Extreme Bear (5th pct)": "red",
        "Base / Median (50th pct)": "black",
        "Extreme Bull (95th pct)": "green",
    }
    for name, price in scenarios.items():
        if name in scenario_colors:
            c = scenario_colors[name]
            ax.plot(future_dates[-1], price, "o", color=c)
            ax.annotate(
                f"{name.split('(')[0].strip()}\n{price:,.2f}",
                xy=(future_dates[-1], price),
                xytext=(10, 0),
                textcoords="offset points",
                fontsize=8,
                color=c,
                va="center",
            )
    ax.axvline(close.index[-1], color="grey", ls=":", lw=1)
    ax.set_title(f"{ticker} — 1-Year Monte Carlo ({n_simulations:,} simulations, GBM)")
    ax.set_ylabel("Price (USD)")
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
run_full_analysis("NVDA")

In [ ]:
run_full_analysis("ASML")

---
## Benchmark Comparison — vs. Nasdaq 100 (QQQ)

To answer whether any of the three stocks is a "better" investment than the index, we compare them head-to-head using four quantitative metrics derived from the same historical window:

| Metric | What it tells us |
|---|---|
| **Ann. Return** | Average yearly gain (drift) implied by historical prices |
| **Ann. Volatility** | Risk — how much prices swing year-to-year |
| **Sharpe Ratio** | Return per unit of risk (higher = better, risk-free rate ≈ 4.3%) |
| **Max Drawdown** | Worst peak-to-trough loss observed — measures downside pain |

A stock beats the index only if it offers meaningfully higher risk-adjusted return (Sharpe), not just higher raw return.

In [ ]:
RISK_FREE_RATE = 0.043  # ~US 10Y yield as of mid-2026


def get_metrics(ticker, start="2020-01-01", end="2026-05-11"):
    raw = yf.download(tickers=ticker, start=start, end=end, progress=False)
    close = raw["Close"].squeeze()
    log_r = np.log(close / close.shift(1)).dropna()

    ann_ret = log_r.mean() * 252
    ann_vol = log_r.std() * np.sqrt(252)
    sharpe = (ann_ret - RISK_FREE_RATE) / ann_vol

    # Max drawdown
    roll_max = close.cummax()
    drawdown = (close - roll_max) / roll_max
    max_dd = drawdown.min()

    # Monte Carlo median 1-year price
    np.random.seed(42)
    n = 252
    drift = log_r.mean() - 0.5 * log_r.std() ** 2
    shocks = log_r.std() * np.random.normal(0, 1, size=(n, 1000))
    paths = np.zeros((n + 1, 1000))
    paths[0] = close.iloc[-1]
    for t in range(1, n + 1):
        paths[t] = paths[t - 1] * np.exp(drift + shocks[t - 1])

    return {
        "Current Price": close.iloc[-1],
        "Ann. Return": ann_ret,
        "Ann. Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd,
        "MC Median 1Y": np.median(paths[-1]),
        "MC Bear 1Y (5th)": np.percentile(paths[-1], 5),
        "MC Bull 1Y (95th)": np.percentile(paths[-1], 95),
    }


tickers = {
    "SK Hynix (000660.KS)": "000660.KS",
    "NVIDIA (NVDA)": "NVDA",
    "ASML (ASML)": "ASML",
    "Nasdaq 100 (QQQ)": "QQQ",
}

results = {name: get_metrics(t) for name, t in tickers.items()}
df_cmp = pd.DataFrame(results).T

# ── Pretty print ─────────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print(
    f"  Head-to-Head Comparison  (2020-01-01 → 2026-05-11, RF = {RISK_FREE_RATE:.1%})"
)
print(f"{'='*72}")
fmt = {
    "Current Price": "{:>12,.2f}",
    "Ann. Return": "{:>11.1%}",
    "Ann. Volatility": "{:>11.1%}",
    "Sharpe Ratio": "{:>11.2f}",
    "Max Drawdown": "{:>11.1%}",
    "MC Median 1Y": "{:>12,.2f}",
    "MC Bear 1Y (5th)": "{:>12,.2f}",
    "MC Bull 1Y (95th)": "{:>12,.2f}",
}
header = f"  {'Metric':<22}" + "".join(f"{n:>20}" for n in results)
print(header)
print(f"  {'-'*70}")
for metric, f in fmt.items():
    row = f"  {metric:<22}"
    for name in results:
        val = results[name][metric]
        row += f"{f.format(val):>20}"
    print(row)

# ── Bar chart: Sharpe Ratio comparison ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics_to_plot = ["Ann. Return", "Ann. Volatility", "Sharpe Ratio"]
colors_bar = ["steelblue", "orange", "green", "red"]
labels = list(results.keys())

for ax, metric in zip(axes, metrics_to_plot):
    vals = [results[n][metric] for n in labels]
    bars = ax.bar(labels, vals, color=colors_bar, edgecolor="black", linewidth=0.5)
    ax.set_title(metric)
    ax.set_ylabel("%" if "Return" in metric or "Volatility" in metric else "")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)
    for bar, v in zip(bars, vals):
        label = (
            f"{v:.1%}" if "Return" in metric or "Volatility" in metric else f"{v:.2f}"
        )
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + abs(max(vals)) * 0.01,
            label,
            ha="center",
            va="bottom",
            fontsize=9,
        )
    # Highlight QQQ as benchmark
    bars[-1].set_edgecolor("black")
    bars[-1].set_linewidth(2)

plt.subtitle(
    "Individual Stocks vs. Nasdaq 100 (QQQ) — Key Metrics", fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

# ── Verdict ───────────────────────────────────────────────────────────────────
qqq_sharpe = results["Nasdaq 100 (QQQ)"]["Sharpe Ratio"]
print(f"\n{'='*60}")
print("  VERDICT")
print(f"{'='*60}")
for name in ["SK Hynix (000660.KS)", "NVIDIA (NVDA)", "ASML (ASML)"]:
    s = results[name]["Sharpe Ratio"]
    dd = results[name]["Max Drawdown"]
    ret = results[name]["Ann. Return"]
    qqq_ret = results["Nasdaq 100 (QQQ)"]["Ann. Return"]
    qqq_dd = results["Nasdaq 100 (QQQ)"]["Max Drawdown"]
    beats = s > qqq_sharpe
    verdict = (
        "BEATS QQQ (risk-adjusted)" if beats else "UNDERPERFORMS QQQ (risk-adjusted)"
    )
    print(f"\n  {name}")
    print(f"    Sharpe {s:.2f} vs QQQ {qqq_sharpe:.2f}  →  {verdict}")
    print(
        f"    Ann. Return {ret:.1%} | Max DD {dd:.1%}  (QQQ: {qqq_ret:.1%} | {qqq_dd:.1%})"
    )
print(f"\n  Note: past Sharpe ratio does NOT guarantee future outperforming.")
print(
    f"        Higher individual stock volatility means higher risk of large dropdowns."
)